Following code section is used to process the raw audio.

In [ ]:
import os
import pandas as pd

def collect_mp3_files(folder: str, recursive: bool = True):
    """Collect all mp3 files from the given folder."""
    mp3_files = []
    if recursive:
        for r, _, fs in os.walk(folder):
            for f in fs:
                if f.lower().endswith(".mp3"):
                    mp3_files.append(os.path.join(r, f))
    else:
        mp3_files = [
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.lower().endswith(".mp3")
        ]
    return sorted(mp3_files)

def extract_acoustic_features(mp3_files: list, sr: int):
    """Extract acoustic statistics for each mp3 file."""
    rows = []
    for i, p in enumerate(mp3_files, 1):
        try:
            d = compute_acoustic_stats(p, sr=sr)
            d["ok"] = True
            d["error"] = ""
        except Exception as e:
            d = {"path": p, "sr": sr, "ok": False, "error": repr(e)}
        rows.append(d)

        if i % 50 == 0 or i == len(mp3_files):
            ok = sum(r.get("ok", False) for r in rows)
            print(f"[batch] {i}/{len(mp3_files)} done | ok={ok} fail={i-ok}")
    return rows

def save_to_file(df: pd.DataFrame, out_path: str):
    """Save the dataframe to a specified file format."""
    ext = os.path.splitext(out_path)[1].lower()
    if ext in [".parquet", ".pq"]:
        df.to_parquet(out_path, index=False)
    else:
        df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"[batch] saved -> {out_path}")

def parse_file_ids(df: pd.DataFrame):
    """Parse dialogue and utterance ids from filenames."""
    base = df["path"].map(lambda x: os.path.basename(x) if isinstance(x, str) else "")
    df["basename"] = base

    def _parse_ids(name: str):
        try:
            stem = os.path.splitext(name)[0]
            dia = int(stem.split("_")[0].replace("dia", ""))
            utt = int(stem.split("_")[1].replace("utt", ""))
            return dia, utt
        except Exception:
            return None, None

    ids = base.map(_parse_ids)
    df["dialogue_id"] = ids.map(lambda t: t[0])
    df["utterance_id"] = ids.map(lambda t: t[1])

def batch_folder_mp3(folder: str, out_path: str = "prosody_stats.csv", sr: int = 16000, recursive: bool = True):
    """Process all mp3 files in the folder and extract prosody statistics."""
    mp3_files = collect_mp3_files(folder, recursive)
    print(f"[batch] found {len(mp3_files)} mp3 files under: {folder}")

    rows = extract_acoustic_features(mp3_files, sr)
    df = pd.DataFrame(rows)

    parse_file_ids(df)
    save_to_file(df, out_path)

    fail_path = os.path.splitext(out_path)[0] + ".failed.csv"
    df[df["ok"] == False][["path", "error"]].to_csv(fail_path, index=False, encoding="utf-8-sig")
    print(f"[batch] failed list -> {fail_path}")

    return df

df = batch_folder_mp3("/root/autodl-tmp/MELD/dev_splits_complete", "meld_dev_prosody_for_test.csv", sr=16000, recursive=True)
df.head()

In [ ]:
import pandas as pd
import numpy as np

# ===== 1️⃣ Read Data =====
df = pd.read_csv("/root/autodl-tmp/MELD/meld_test_prosody.csv")

# ===== 2️⃣ Handle Missing F0 =====
df.loc[df["voiced_ratio"] == 0, ["f0_std_hz", "f0_slope_hz_per_sec"]] = np.nan

# ===== 3️⃣ Quantile Binning Function =====
def quantile_bin(series):
    q1 = series.quantile(0.33)
    q2 = series.quantile(0.66)

    def label(x):
        if pd.isna(x):
            return "missing"
        elif x <= q1:
            return "low"
        elif x <= q2:
            return "mid"
        else:
            return "high"

    return series.apply(label)

# ===== 4️⃣ Discretize Core Features =====
df["loudness_bin"] = quantile_bin(df["rms_mean"])
df["loudness_dynamics_bin"] = quantile_bin(df["rms_range_p95_p05"])
df["pitch_variability_bin"] = quantile_bin(df["f0_std_hz"])
df["voicing_bin"] = quantile_bin(df["voiced_ratio"])
df["rate_bin"] = quantile_bin(df["rate_proxy_peaks_per_sec"])

# ===== 5️⃣ Process Slope Separately =====
def slope_bin(x):
    if pd.isna(x):
        return "missing"
    elif x < -10:
        return "falling"
    elif x > 10:
        return "rising"
    else:
        return "flat"

df["pitch_trend"] = df["f0_slope_hz_per_sec"].apply(slope_bin)

# ===== 6️⃣ Simple Duration Discretization (Optional) =====
dur_q1 = df["duration_sec"].quantile(0.33)
dur_q2 = df["duration_sec"].quantile(0.66)

def duration_bin(x):
    if x <= dur_q1:
        return "short"
    elif x <= dur_q2:
        return "mid"
    else:
        return "long"

df["duration_bin"] = df["duration_sec"].apply(duration_bin)

# ===== 7️⃣ Print Distribution for Each Bin (for Checking) =====
cols_to_check = [
    "loudness_bin",
    "loudness_dynamics_bin",
    "pitch_variability_bin",
    "voicing_bin",
    "rate_bin",
    "pitch_trend",
    "duration_bin"
]

for col in cols_to_check:
    print(f"\n=== {col} ===")
    print(df[col].value_counts())

# ===== 8️⃣ Save New File =====
df.to_csv("test_audio_features_binned.csv", index=False)

print("\n✅ Discretization completed, file saved as audio_features_binned.csv")